# Embrix Python SDK — Full Demo



This notebook exercises the **PyPI package** end-to-end using the local repository source.



Covered areas:

- provider-agnostic chat model setup

- vector store search

- state graph workflows

- ReAct agent with tools

- RAG pipeline

- database vectorization + natural-language querying



Recommended kernel: the existing project venv at `embrix-chain/backend/.venv/bin/python` (it already has `numpy`, which the Python package needs).



> Security note: the notebook reads `OPENAI_API_KEY` from your environment or the project-root `.env` file. Do **not** paste secrets directly into notebook cells.


In [3]:
from __future__ import annotations



import hashlib

import json

import os

import sqlite3

import sys

from pathlib import Path



ROOT = Path('/Users/hemantkumarthapa/Desktop/RAG/VB')

if str(ROOT) not in sys.path:

    sys.path.insert(0, str(ROOT))



def load_dotenv(dotenv_path: Path) -> None:

    if not dotenv_path.exists():

        return

    for raw_line in dotenv_path.read_text().splitlines():

        line = raw_line.strip()

        if not line or line.startswith('#') or '=' not in line:

            continue

        key, value = line.split('=', 1)

        key = key.strip()

        value = value.strip().strip('"').strip("'")

        if key and value and key not in os.environ:

            os.environ[key] = value



def load_local_api_key(key_name: str, fallback_file: Path) -> None:

    if os.environ.get(key_name):

        return

    if fallback_file.exists():

        value = fallback_file.read_text().strip()

        if value:

            os.environ[key_name] = value



load_dotenv(ROOT / '.env')

load_local_api_key('OPENAI_API_KEY', ROOT / 'openai-api')



try:

    import numpy as np  # noqa: F401

except ImportError as exc:

    raise RuntimeError(

        'This notebook needs a Python environment with numpy installed. '

        'Use the embrix-chain/backend/.venv kernel or install project dependencies first.'

    ) from exc



def require_env(name: str) -> str:

    value = os.environ.get(name, '').strip()

    if not value:

        raise RuntimeError(f'Missing {name}. Set it in your shell, the project-root .env file, or the local openai-api file.')

    return value



print('Root:', ROOT)

print('Python executable:', sys.executable)

print('OPENAI_API_KEY loaded:', bool(os.environ.get('OPENAI_API_KEY')))


Root: /Users/hemantkumarthapa/Desktop/RAG/VB
Python executable: /Users/hemantkumarthapa/Desktop/RAG/VB/embrix-chain/backend/.venv/bin/python
OPENAI_API_KEY loaded: True


In [2]:
from embrix import (
    Agent,
    DBQueryEngine,
    DBVectorizer,
    END,
    OpenAICompatibleChatModel,
    RAGPipeline,
    START,
    StateGraph,
    Tool,
    VectorStore,
    create_chat_model,
)
from embrix.db import connect as db_connect
from embrix.embeddings.base import BaseEmbedder

class HashEmbedder(BaseEmbedder):
    def __init__(self, dimension: int = 16):
        self._dimension = dimension

    @property
    def dimension(self) -> int:
        return self._dimension

    def embed(self, texts: list[str]) -> list[list[float]]:
        vectors: list[list[float]] = []
        for text in texts:
            tokens = text.lower().split()
            vec = [0.0] * self._dimension
            for token in tokens:
                digest = hashlib.sha256(token.encode('utf-8')).digest()
                idx = digest[0] % self._dimension
                vec[idx] += 1.0
            norm = sum(value * value for value in vec) ** 0.5 or 1.0
            vectors.append([value / norm for value in vec])
        return vectors

chat_model = OpenAICompatibleChatModel(
    model=os.environ.get('OPENAI_MODEL', 'gpt-4o-mini'),
    api_key=require_env('OPENAI_API_KEY'),
    base_url=os.environ.get('OPENAI_BASE_URL') or None,
    system_prompt='You are a precise demo assistant. Keep answers short but correct.',
)

hash_embedder = HashEmbedder()
print('Python demo ready.')

RuntimeError: Missing OPENAI_API_KEY. Set it in your shell or the project-root .env file.

In [ ]:
# 1) Local vector store demo
vector_store = VectorStore(backend='memory', dimension=hash_embedder.dimension)
records = [
    {'_id': 'doc-1', 'values': hash_embedder.embed_one('embrix is a local first AI infrastructure library'), 'text': 'embrix is a local first AI infrastructure library', 'metadata': {'topic': 'overview'}},
    {'_id': 'doc-2', 'values': hash_embedder.embed_one('langgraph style state graphs orchestrate workflows'), 'text': 'langgraph style state graphs orchestrate workflows', 'metadata': {'topic': 'graph'}},
    {'_id': 'doc-3', 'values': hash_embedder.embed_one('vector databases power semantic retrieval for RAG'), 'text': 'vector databases power semantic retrieval for RAG', 'metadata': {'topic': 'rag'}},
]
vector_store.upsert('demo', records)
results = vector_store.query('demo', hash_embedder.embed_one('semantic retrieval for rag'), top_k=2)
results

In [ ]:
# 2) StateGraph demo

def route_question(state: dict) -> str:

    return {'needs_rag': 'retrieve_and_answer', 'simple': 'answer_simple'}[state['mode']]



def retrieve_and_answer(state: dict) -> dict:

    hits = vector_store.query('demo', hash_embedder.embed_one(state['question']), top_k=2)

    context = '\n'.join(hit['text'] for hit in hits)

    answer = chat_model.invoke(

        f'Use this context to answer briefly.\nContext:\n{context}\n\nQuestion: {state["question"]}'

    )

    return {'hits': hits, 'answer': answer}



def answer_simple(state: dict) -> dict:

    return {'answer': f"Simple path handled: {state['question']}"}



def finish(state: dict) -> dict:

    return {'finished': True}



graph = StateGraph()

graph.add_node('route', lambda state: state)

graph.add_node('retrieve_and_answer', retrieve_and_answer)

graph.add_node('answer_simple', answer_simple)

graph.add_node('finish', finish)

graph.add_edge(START, 'route')

graph.add_conditional_edges('route', route_question)

graph.add_edge('retrieve_and_answer', 'finish')

graph.add_edge('answer_simple', 'finish')

graph.add_edge('finish', END)

compiled = graph.compile()

compiled.invoke({'mode': 'needs_rag', 'question': 'What does embrix help with?'})


In [ ]:
# 3) Agent demo with a real OpenAI-compatible model
def calculator(expression: str) -> str:
    allowed = set('0123456789+-*/(). ')
    if not set(expression) <= allowed:
        raise ValueError('Unsupported characters in expression')
    return str(eval(expression, {'__builtins__': {}}, {}))

agent = Agent(
    tools=[Tool('calculator', 'Evaluate arithmetic expressions', calculator)],
    llm=chat_model,
    verbose=False,
)
agent.run('Use the calculator tool to compute (19 * 7) + 5, then explain the result in one sentence.')

In [ ]:
# 4) RAGPipeline demo
rag = RAGPipeline(embedder=hash_embedder, llm=chat_model, store=VectorStore(backend='memory', dimension=hash_embedder.dimension))
rag.ingest([
    'Embrix combines local vector search, agents, workflows, and database tooling.',
    'The Python package is ideal for notebooks, local demos, and backend integration.',
    'RAG pipelines retrieve chunks before generation to keep answers grounded.',
], namespace='knowledge')
rag.query('What are the main parts of Embrix?', namespace='knowledge')

In [ ]:
# 5) Database vectorization + DBQueryEngine demo
db_file = ROOT / 'demo_notebooks' / 'embrix_python_demo.sqlite'
if db_file.exists():
    db_file.unlink()

conn = sqlite3.connect(db_file)
conn.execute('CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL, description TEXT)')
conn.executemany(
    'INSERT INTO products (name, category, price, description) VALUES (?, ?, ?, ?)',
    [
        ('Trail Runner Pro', 'shoes', 89.0, 'lightweight running shoe for trails and long distance'),
        ('Studio Comfort', 'headphones', 129.0, 'over ear headphones with clear sound and strong bass'),
        ('City Sprint', 'shoes', 59.0, 'everyday running shoe for short road runs'),
    ],
)
conn.commit()
conn.close()

sqlite_connector = db_connect('sqlite', db_path=str(db_file))
vectorizer = DBVectorizer(sqlite_connector, hash_embedder, store=VectorStore(backend='memory', dimension=hash_embedder.dimension))
vector_count = vectorizer.vectorize_table('products', namespace='products_demo', text_columns=['name', 'description', 'category'])
engine = DBQueryEngine(vectorizer=vectorizer, llm=chat_model, connector=sqlite_connector)
semantic_hits = engine.search('cheap running shoes', namespace='products_demo', top_k=2)
rag_answer = engine.query('Which product is best for someone wanting running shoes?', namespace='products_demo', top_k=2)
sql_rows = engine.sql_query('Show the names and prices of products under 100 dollars', table='products')
{
    'vector_count': vector_count,
    'semantic_hits': semantic_hits,
    'rag_answer': rag_answer,
    'sql_rows': sql_rows,
}

## Next ideas

Try swapping the provider without changing the rest of the code:

```python
anthropic = create_chat_model('anthropic', model='claude-3-5-sonnet-latest', api_key=os.environ['ANTHROPIC_API_KEY'])
gemini = create_chat_model('gemini', model='gemini-1.5-flash', api_key=os.environ['GEMINI_API_KEY'])
ollama = create_chat_model('ollama', model='llama3.1')
```